# Chapter 2 — Tensor Operations for NLP (Practice)

Work through these exercises **after reading** `notes/ch02-tensor-ops-for-nlp.md`.

Each exercise states the *decision you're practicing*, gives a stub cell to fill in, and is followed by a pre-written **verification cell** — run it to grade yourself. Don't peek at `solutions/` until the verification passes or you're genuinely stuck.

In [1]:
# ============================================================
# TOPIC: The ~15 tensor ops that are 90% of LLM code
# MATH:  softmax(z)_i = exp(z_i/T) / sum_j exp(z_j/T);  C_ij = sum_k A_ik B_kj
# REF:   B00 ch02 notes — tensor-ops-for-nlp
# ============================================================

# --- Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Reproducibility & device ---
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version : {torch.__version__}")
print(f"device        : {device}  (every exercise here runs fine on CPU)")

torch version : 2.13.0
device        : cpu  (every exercise here runs fine on CPU)


## Exercise 1 — One Computation, Three Matmuls

Compute batched multi-head attention scores `(B, H, T, T)` from `queries` and `keys` three different ways. All three must produce identical numbers — the point is to *feel* the bookkeeping each variant demands.

1. `scores_matmul` — plain `@` (batch dims broadcast for free)
2. `scores_bmm` — `torch.bmm`, which only accepts 3-D, so you must do the fold/unfold reshape dance
3. `scores_einsum` — one index string, no explicit transpose

**Decision you're practicing:** the matmul-family guide — default to `@`, reach for strict ops or `einsum` deliberately.

In [2]:
B, H, T, D_HEAD = 2, 3, 4, 5
queries = torch.randn(B, H, T, D_HEAD)   # shape: (batch, heads, seq, head_dim)
keys    = torch.randn(B, H, T, D_HEAD)

def scores_matmul(q, k):
    """(B, H, T, d) x (B, H, T, d) -> (B, H, T, T) using @."""
    return q @ k.transpose(-2, -1)                      # batch dims (B, H) ride along for free

def scores_bmm(q, k):
    """Same result using torch.bmm (strict 3-D: fold B*H together, unfold after)."""
    batch, heads, seq, head_dim = q.shape
    q_folded = q.reshape(batch * heads, seq, head_dim)  # (B*H, T, d) — bmm demands exactly 3-D
    k_folded = k.reshape(batch * heads, seq, head_dim)
    scores = torch.bmm(q_folded, k_folded.transpose(1, 2))   # (B*H, T, T)
    return scores.reshape(batch, heads, seq, seq)       # unfold back to (B, H, T, T)

def scores_einsum(q, k):
    """Same result using one torch.einsum call."""
    return torch.einsum("bhqd,bhkd->bhqk", q, k)        # d vanishes → summed; transpose implicit

print(f"@      bookkeeping: 1 transpose")
print(f"bmm    bookkeeping: 2 reshapes in, 1 transpose, 1 reshape out — strictness has a price")
print(f"einsum bookkeeping: 1 string ('bhqd,bhkd->bhqk')")

@      bookkeeping: 1 transpose
bmm    bookkeeping: 2 reshapes in, 1 transpose, 1 reshape out — strictness has a price
einsum bookkeeping: 1 string ('bhqd,bhkd->bhqk')


**Verification**

In [3]:
# --- Verification: Exercise 1 ---
results = {
    "scores_matmul": scores_matmul(queries, keys),
    "scores_bmm":    scores_bmm(queries, keys),
    "scores_einsum": scores_einsum(queries, keys),
}
reference = results["scores_matmul"]
assert reference is not None, "fill in the stubs above first"
for name, scores in results.items():
    assert scores is not None, f"{name} not implemented yet"
    assert scores.shape == (B, H, T, T), f"{name}: wrong shape {tuple(scores.shape)}"
    assert torch.allclose(scores, reference, atol=1e-5), f"{name}: numbers differ!"
    print(f"{name:14s} → {tuple(scores.shape)} ✓ identical")
print("Exercise 1 passed ✓")

scores_matmul  → (2, 3, 4, 4) ✓ identical
scores_bmm     → (2, 3, 4, 4) ✓ identical
scores_einsum  → (2, 3, 4, 4) ✓ identical
Exercise 1 passed ✓


## Exercise 2 — The Wrong-dim Softmax

The buggy line below shipped to prod: attention weights over `(batch, T_query, T_key)` scores, normalized along `dim=1`. It doesn't crash. It just trains badly.

Diagnose it with prints (which axis sums to 1? what does that *mean* was normalized?), fix it, and state the rule in `the_rule`.

**Decision you're practicing:** `dim` = the axis whose entries compete — and how to *detect* a wrong-dim softmax after the fact.

In [4]:
torch.manual_seed(7)
raw_scores = torch.randn(2, 4, 6)                            # shape: (batch, T_query=4, T_key=6)

attention_weights_buggy = torch.softmax(raw_scores, dim=1)   # ← the bug

print("buggy  .sum(dim=1)[0] :", [round(v, 3) for v in attention_weights_buggy.sum(dim=1)[0].tolist()])
print("buggy  .sum(dim=-1)[0]:", [round(v, 3) for v in attention_weights_buggy.sum(dim=-1)[0].tolist()])
print("→ dim=1 is all-ones: the 4 QUERIES were made to compete for each key.")
print("→ Semantically nonsense — each query's attention over keys should sum to 1.\n")

attention_weights_fixed = torch.softmax(raw_scores, dim=-1)
print("fixed  .sum(dim=-1)[0]:", [round(v, 3) for v in attention_weights_fixed.sum(dim=-1)[0].tolist()])

the_rule = ("softmax's dim is the axis whose entries compete for probability mass; "
            "for (batch, T_q, T_k) attention scores the KEYS compete per query, so dim=-1")
print(f"\nRule: {the_rule}")

buggy  .sum(dim=1)[0] : [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
buggy  .sum(dim=-1)[0]: [1.314, 0.84, 1.661, 2.185]
→ dim=1 is all-ones: the 4 QUERIES were made to compete for each key.
→ Semantically nonsense — each query's attention over keys should sum to 1.

fixed  .sum(dim=-1)[0]: [1.0, 1.0, 1.0, 1.0]

Rule: softmax's dim is the axis whose entries compete for probability mass; for (batch, T_q, T_k) attention scores the KEYS compete per query, so dim=-1


**Verification**

In [5]:
# --- Verification: Exercise 2 ---
assert attention_weights_fixed is not None, "fill in the stub above first"
key_axis_sums = attention_weights_fixed.sum(dim=-1)
assert torch.allclose(key_axis_sums, torch.ones_like(key_axis_sums), atol=1e-5), \
    "each query's weights over the keys must sum to 1"
assert not torch.allclose(attention_weights_buggy.sum(dim=-1), torch.ones(2, 4), atol=1e-3), \
    "sanity: the buggy version is NOT normalized along the key axis"
assert torch.allclose(attention_weights_fixed, torch.softmax(raw_scores, dim=-1))
assert isinstance(the_rule, str) and len(the_rule) > 20, "state the rule in your own words"
print(f"your rule: {the_rule}")
print("Exercise 2 passed ✓")

your rule: softmax's dim is the axis whose entries compete for probability mass; for (batch, T_q, T_k) attention scores the KEYS compete per query, so dim=-1
Exercise 2 passed ✓


## Exercise 3 — Cross-Entropy by Hand, with `gather`

Implement cross-entropy from scratch: `log_softmax` → `gather` the correct-class log-probs → mask out `ignore_index` positions → mean over the kept ones. Then match PyTorch's official answer to 6 decimals.

**Decision you're practicing:** `gather` as "batched per-row lookup" — the op hiding inside every language-model loss — plus boolean masking of the ignored positions.

In [6]:
def manual_cross_entropy(logits, targets, ignore_index=-100):
    """Cross-entropy loss, averaged over non-ignored positions.

    Args:
        logits: (batch, vocab) float tensor of raw scores
        targets: (batch,) int64 tensor; positions equal to ignore_index don't count
    Returns:
        scalar loss tensor.
    Math note: loss = -mean over kept i of log_softmax(logits)[i, targets[i]]
    """
    log_probs = F.log_softmax(logits, dim=-1)                    # (batch, vocab)
    keep = targets != ignore_index                               # (batch,) bool — real positions
    print(f"log_probs {tuple(log_probs.shape)}, keeping {int(keep.sum())}/{len(targets)} positions")

    safe_targets = targets.clone()
    safe_targets[~keep] = 0                                      # any valid index — gather can't take -100

    picked = log_probs.gather(dim=1, index=safe_targets.unsqueeze(1)).squeeze(1)   # (batch,)
    loss = -picked[keep].mean()                                  # boolean indexing drops ignored rows
    print(f"correct-class log-probs (kept): {[round(v, 3) for v in picked[keep].tolist()]}")
    return loss

**Verification**

In [7]:
# --- Verification: Exercise 3 ---
torch.manual_seed(11)
test_logits = torch.randn(6, 9)                          # shape: (batch=6, vocab=9)
test_targets = torch.tensor([3, -100, 0, 8, -100, 5])    # two ignored positions

mine = manual_cross_entropy(test_logits, test_targets)
assert mine is not None, "fill in the stub above first"
official = F.cross_entropy(test_logits, test_targets, ignore_index=-100)
print(f"manual   : {mine.item():.6f}")
print(f"official : {official.item():.6f}")
assert torch.allclose(mine, official, atol=1e-6), "loss mismatch — check the masking of ignored rows"
print("Exercise 3 passed ✓")

log_probs (6, 9), keeping 4/6 positions
correct-class log-probs (kept): [-2.053, -4.137, -2.681, -1.387]
manual   : 2.564503
official : 2.564503
Exercise 3 passed ✓


## Exercise 4 — Causal Mask from Scratch

Build causal attention weights for a `(T, T)` score matrix: token `t` may only attend to positions `≤ t`. Construct the mask with `torch.triu`, apply it with `masked_fill(-inf)`, softmax along the right dim.

**Decision you're practicing:** why the mask goes in *before* softmax with $-\infty$ (survivors renormalize; blocked slots get exactly zero).

In [8]:
def causal_attention_weights(scores):
    """(T, T) raw scores -> (T, T) attention weights with the future blocked.

    Row t must be a valid distribution over positions 0..t and EXACTLY zero after.
    """
    seq_len = scores.shape[-1]
    future_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
    print(f"future_mask (True = block):\n{future_mask}")
    masked_scores = scores.masked_fill(future_mask, float("-inf"))
    print(f"masked scores:\n{masked_scores}")
    weights = torch.softmax(masked_scores, dim=-1)       # exp(-inf)=0; survivors renormalize
    return weights

**Verification**

In [9]:
# --- Verification: Exercise 4 ---
torch.manual_seed(5)
test_scores = torch.randn(4, 4)
weights = causal_attention_weights(test_scores)
assert weights is not None, "fill in the stub above first"
assert weights.shape == (4, 4)

strictly_upper = torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)
assert (weights[strictly_upper] == 0).all(), "future positions must be EXACTLY zero (not just tiny)"
assert torch.allclose(weights.sum(dim=-1), torch.ones(4), atol=1e-6), "each row must still sum to 1"

# loop reference: row t = softmax over the visible prefix only
for row in range(4):
    expected_prefix = torch.softmax(test_scores[row, : row + 1], dim=-1)
    assert torch.allclose(weights[row, : row + 1], expected_prefix, atol=1e-6), f"row {row} mismatch"
    print(f"row {row}: attends to {row + 1} position(s) → {[round(v, 3) for v in weights[row, : row + 1].tolist()]} ✓")
print("Exercise 4 passed ✓")

future_mask (True = block):
tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])
masked scores:
tensor([[ 1.8423,    -inf,    -inf,    -inf],
        [ 2.0194, -0.2686,    -inf,    -inf],
        [ 0.3908, -0.0190, -1.3527,    -inf],
        [ 0.9879, -0.4194, -0.5849, -0.7823]])
row 0: attends to 1 position(s) → [1.0] ✓
row 1: attends to 2 position(s) → [0.908, 0.092] ✓
row 2: attends to 3 position(s) → [0.544, 0.361, 0.095] ✓
row 3: attends to 4 position(s) → [0.616, 0.151, 0.128, 0.105] ✓
Exercise 4 passed ✓


## Exercise 5 — Choose the Masking Tool

Three mini-tasks, each with a constraint that makes exactly one tool from notes §5 the right call. Fill the outputs **and** record which tool you used in `technique` (allowed values: `"multiply"`, `"masked_fill"`, `"boolean_indexing"`, `"where"`).

1. `zeroed` — pad-token embeddings must become zero vectors (they feed a *sum* later, so zero is neutral)
2. `max_scores` — the max score per sequence over **real positions only** (careful: row 0's real scores are all negative, and a pad slot holds `2.0`)
3. `real_vectors` — all real-token embeddings extracted into one flat `(n_real, dim)` tensor

**Decision you're practicing:** choosing the mask tool by *what happens downstream* — not by habit.

In [10]:
token_embeddings = torch.randn(2, 4, 3)      # shape: (batch, seq, dim)
position_scores = torch.tensor([
    [-0.5, -1.2, -0.3,  2.0],                # row 0: real scores all NEGATIVE; the 2.0 sits on a pad!
    [ 0.7, -0.8,  1.5, -2.0],
])                                           # shape: (batch, seq)
real_mask = torch.tensor([
    [True, True,  True,  False],
    [True, False, False, False],
])                                           # True = real token

# (1) zero is neutral for the downstream sum → cheapest tool wins: multiply
zeroed = token_embeddings * real_mask.unsqueeze(-1)          # (2,4,3) * (2,4,1) → broadcast

# (2) a zeroed pad would BEAT row 0's all-negative real scores → must use -inf, not 0
max_scores = position_scores.masked_fill(~real_mask, float("-inf")).max(dim=1).values
wrong_way = (position_scores * real_mask).max(dim=1).values  # the multiply mistake, for contrast
print(f"masked_fill way: {max_scores.tolist()}   ← correct: [-0.3, 0.7]")
print(f"multiply way   : {wrong_way.tolist()}   ← row 0 wrong: the zeroed pad beat every negative real score")

# (3) downstream code should never even see pads → remove them entirely
real_vectors = token_embeddings[real_mask]                   # (n_real=4, dim=3), flattened
print(f"real_vectors: {tuple(real_vectors.shape)}  ({int(real_mask.sum())} real tokens)")

technique = {
    "zero_pad_embeddings": "multiply",
    "max_over_real":       "masked_fill",
    "flatten_real_tokens": "boolean_indexing",
}

masked_fill way: [-0.30000001192092896, 0.699999988079071]   ← correct: [-0.3, 0.7]
multiply way   : [0.0, 0.699999988079071]   ← row 0 wrong: the zeroed pad beat every negative real score
real_vectors: (4, 3)  (4 real tokens)


**Verification**

In [11]:
# --- Verification: Exercise 5 ---
assert zeroed is not None and max_scores is not None and real_vectors is not None, "fill in the stubs first"

# (1) pad rows zero, real rows untouched
assert (zeroed[~real_mask] == 0).all(), "pad positions must be exactly zero"
assert torch.equal(zeroed[real_mask], token_embeddings[real_mask]), "real positions must be untouched"

# (2) max over real positions only — row 0 catches the multiply-by-mask mistake
expected_max = torch.tensor([-0.3, 0.7])
assert torch.allclose(max_scores, expected_max, atol=1e-6), \
    f"got {max_scores.tolist()} — did a zeroed pad win the max? (0 > all of row 0's real scores)"

# (3) flat real-token embeddings
assert real_vectors.shape == (int(real_mask.sum()), 3), f"wrong shape {tuple(real_vectors.shape)}"
assert torch.equal(real_vectors, token_embeddings[real_mask])

expected_technique = {
    "zero_pad_embeddings": "multiply",
    "max_over_real":       "masked_fill",
    "flatten_real_tokens": "boolean_indexing",
}
for task, expected_tool in expected_technique.items():
    assert technique[task] == expected_tool, \
        f"{task}: expected {expected_tool!r} (revisit the decision table in notes §5), got {technique[task]!r}"
    print(f"✓ {task:22s} → {expected_tool}")
print("Exercise 5 passed ✓")

✓ zero_pad_embeddings    → multiply
✓ max_over_real          → masked_fill
✓ flatten_real_tokens    → boolean_indexing
Exercise 5 passed ✓


## Exercise 6 — Temperature + Top-k Sampler

Implement `sample_next_token(logits, top_k, temperature)` as the four-step pipeline from notes §6:

```
divide by temperature → keep top-k (mask the rest to -inf) → softmax → multinomial
```

The verification draws 500 samples at three temperatures and checks the statistics: every sample inside the top-k set, `T=0.1` almost always picks the argmax, `T=5.0` spreads across all k survivors.

**Decision you're practicing:** greedy vs sampling ops — and why `multinomial` needs probabilities, never raw logits.

In [12]:
def sample_next_token(logits, top_k, temperature):
    """Sample one token id from (vocab,) logits via temperature + top-k.

    Args:
        logits: (vocab,) float tensor of raw scores
        top_k: keep only this many candidates
        temperature: divides the logits BEFORE softmax (T<1 sharper, T>1 flatter)
    Returns:
        int token id.
    Math note: p_i = exp(z_i/T) / sum_j exp(z_j/T), restricted to the top-k set
    """
    scaled = logits / temperature                                  # (vocab,) — reshape the contest
    top_values, _ = torch.topk(scaled, top_k)                      # (top_k,) best values, sorted desc
    threshold = top_values[-1]                                     # the k-th best score
    pruned = scaled.masked_fill(scaled < threshold, float("-inf")) # garbage tail → exactly 0 prob
    probs = torch.softmax(pruned, dim=-1)                          # survivors renormalize
    token_id = torch.multinomial(probs, num_samples=1).item()      # multinomial wants PROBS
    return token_id

**Verification**

In [13]:
# --- Verification: Exercise 6 ---
torch.manual_seed(123)
vocab_logits = torch.tensor([4.0, 3.5, 3.0, 1.0, 0.5, 0.0, -0.5, -1.0, -2.0, -5.0])   # vocab of 10
TOP_K = 3
top_k_set = set(torch.topk(vocab_logits, TOP_K).indices.tolist())
print(f"top-{TOP_K} token ids: {sorted(top_k_set)}")

num_draws = 500
samples_sharp = [sample_next_token(vocab_logits, TOP_K, temperature=0.1) for _ in range(num_draws)]
samples_mid   = [sample_next_token(vocab_logits, TOP_K, temperature=1.0) for _ in range(num_draws)]
samples_flat  = [sample_next_token(vocab_logits, TOP_K, temperature=5.0) for _ in range(num_draws)]

for name, samples in [("T=0.1", samples_sharp), ("T=1.0", samples_mid), ("T=5.0", samples_flat)]:
    assert set(samples) <= top_k_set, f"{name}: sampled outside the top-{TOP_K} set!"
    freq_argmax = samples.count(0) / num_draws
    print(f"{name}: argmax frequency = {freq_argmax:.3f}, distinct tokens = {len(set(samples))}")

freq_sharp = samples_sharp.count(0) / num_draws
freq_flat  = samples_flat.count(0)  / num_draws
assert freq_sharp > 0.95, f"T=0.1 should be near-greedy, got argmax freq {freq_sharp:.3f}"
assert freq_flat < 0.6,   f"T=5.0 should spread the mass, got argmax freq {freq_flat:.3f}"
assert len(set(samples_flat)) == TOP_K, "at T=5.0 all top-k survivors should appear in 500 draws"
print("Exercise 6 passed ✓")

top-3 token ids: [0, 1, 2]
T=0.1: argmax frequency = 0.996, distinct tokens = 2
T=1.0: argmax frequency = 0.500, distinct tokens = 3
T=5.0: argmax frequency = 0.416, distinct tokens = 3
Exercise 6 passed ✓


## Exercise 7 — Fused QKV Split, Two Ways

Real transformers project Q, K, V with **one** big matmul into `(B, T, 3·d_model)`, then split. Implement the split two ways, both returning per-head tensors `(B, H, T, head_dim)`:

1. `split_qkv_chunk` — `chunk(3, dim=-1)` then the view/transpose head-split on each piece
2. `split_qkv_unbind` — `view` to `(B, T, 3, H, head_dim)` then `unbind(dim=2)`

**Decision you're practicing:** `chunk` ("I know the *count*") vs `split` ("I know the *sizes*") vs `unbind` ("dissolve an axis") — plus ch01's view/transpose choreography.

In [14]:
B, T, D_MODEL, N_HEADS = 2, 4, 12, 3
HEAD_DIM = D_MODEL // N_HEADS
fused_qkv = torch.randn(B, T, 3 * D_MODEL)     # shape: (batch, seq, 3*d_model) — one projection output

def split_qkv_chunk(fused):
    """-> (q, k, v), each (B, N_HEADS, T, HEAD_DIM), via chunk."""
    q_flat, k_flat, v_flat = fused.chunk(3, dim=-1)          # 3 × (B, T, D_MODEL) — views, no copy
    def to_heads(tensor_2d_heads):
        batch, seq, d_model = tensor_2d_heads.shape
        return tensor_2d_heads.view(batch, seq, N_HEADS, HEAD_DIM).transpose(1, 2)  # (B, H, T, head_dim)
    return to_heads(q_flat), to_heads(k_flat), to_heads(v_flat)

def split_qkv_unbind(fused):
    """-> (q, k, v), each (B, N_HEADS, T, HEAD_DIM), via view + unbind(dim=2)."""
    batch, seq, _ = fused.shape
    stacked = fused.view(batch, seq, 3, N_HEADS, HEAD_DIM)   # (B, T, 3, H, head_dim) — 3 is the qkv axis
    q, k, v = stacked.unbind(dim=2)                          # dissolve the qkv axis → 3 × (B, T, H, head_dim)
    return q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

q_a, k_a, v_a = split_qkv_chunk(fused_qkv)
print(f"chunk  way: q {tuple(q_a.shape)}  # (batch, heads, seq, head_dim)")
q_b, k_b, v_b = split_qkv_unbind(fused_qkv)
print(f"unbind way: q {tuple(q_b.shape)}")

chunk  way: q (2, 3, 4, 4)  # (batch, heads, seq, head_dim)
unbind way: q (2, 3, 4, 4)


**Verification**

In [15]:
# --- Verification: Exercise 7 ---
chunk_result = split_qkv_chunk(fused_qkv)
unbind_result = split_qkv_unbind(fused_qkv)
assert chunk_result is not None and unbind_result is not None, "fill in the stubs first"

# reference: slice the fused tensor by hand, then head-split
slices = [fused_qkv[..., i * D_MODEL:(i + 1) * D_MODEL] for i in range(3)]
reference = [s.view(B, T, N_HEADS, HEAD_DIM).transpose(1, 2) for s in slices]

for name, triple in [("chunk", chunk_result), ("unbind", unbind_result)]:
    for tensor, ref, letter in zip(triple, reference, "qkv"):
        assert tensor.shape == (B, N_HEADS, T, HEAD_DIM), f"{name} {letter}: wrong shape {tuple(tensor.shape)}"
        assert torch.equal(tensor, ref), f"{name} {letter}: values scrambled — check the axis order in the view"
    print(f"{name:6s} way ✓ all of q, k, v match the hand-sliced reference")
print("Exercise 7 passed ✓")

chunk  way ✓ all of q, k, v match the hand-sliced reference
unbind way ✓ all of q, k, v match the hand-sliced reference
Exercise 7 passed ✓


## Exercise 8 — Label Smoothing with `scatter_`

Build soft targets the way PyTorch does it: the smoothed distribution is

$$y^{\text{smooth}} = (1 - \varepsilon)\,\text{one\_hot}(y) + \frac{\varepsilon}{V}$$

i.e. start every class at $\varepsilon / V$ (a `torch.full`), then `scatter_` the value $1 - \varepsilon + \varepsilon/V$ into the target column (the true class keeps its share of the uniform mass too).

The verification proves your distribution is *exactly* what `F.cross_entropy(..., label_smoothing=eps)` uses internally.

**Decision you're practicing:** `scatter_` as the write-inverse of `gather` — building distributions by writing at indexed positions.

In [16]:
def smoothed_targets(targets, vocab_size, eps):
    """(batch,) int64 targets -> (batch, vocab) smoothed distributions (PyTorch convention).

    Every class starts at eps/V; the target column is set to 1 - eps + eps/V.
    Each row must sum to exactly 1.
    """
    batch_size = targets.shape[0]
    smooth = torch.full((batch_size, vocab_size), eps / vocab_size)      # (batch, vocab) uniform floor
    target_value = 1.0 - eps + eps / vocab_size                          # true class keeps its uniform share
    smooth.scatter_(dim=1, index=targets.unsqueeze(1),                   # write-inverse of gather
                    value=target_value)
    print(f"floor prob = {eps / vocab_size:.4f}, target prob = {target_value:.4f}")
    return smooth

**Verification**

In [17]:
# --- Verification: Exercise 8 ---
EPS, VOCAB = 0.1, 8
torch.manual_seed(3)
test_logits = torch.randn(5, VOCAB)                  # shape: (batch=5, vocab=8)
test_targets = torch.randint(0, VOCAB, (5,))

soft = smoothed_targets(test_targets, VOCAB, EPS)
assert soft is not None, "fill in the stub above first"
assert soft.shape == (5, VOCAB)
assert torch.allclose(soft.sum(dim=-1), torch.ones(5), atol=1e-6), "each row must be a distribution"

expected_target_prob = 1 - EPS + EPS / VOCAB
actual_target_probs = soft[torch.arange(5), test_targets]        # ex-3's fancy-index idiom, reused
assert torch.allclose(actual_target_probs, torch.full((5,), expected_target_prob)), \
    f"target column should hold {expected_target_prob:.4f}"

manual_loss = -(soft * F.log_softmax(test_logits, dim=-1)).sum(dim=-1).mean()
official = F.cross_entropy(test_logits, test_targets, label_smoothing=EPS)
print(f"manual soft-target CE : {manual_loss.item():.6f}")
print(f"F.cross_entropy(ls)   : {official.item():.6f}")
assert torch.allclose(manual_loss, official, atol=1e-6), "distributions differ from PyTorch's convention"
print("Exercise 8 passed ✓")

floor prob = 0.0125, target prob = 0.9125
manual soft-target CE : 2.479288
F.cross_entropy(ls)   : 2.479288
Exercise 8 passed ✓


---
## Done!

Compare your work against `solutions/ch02-tensor-ops-for-nlp-solution.ipynb`, then move on to **ch03 — The Autograd Mental Model**. Notice how exercise 6's sampler was assembled entirely from this chapter's ops — that composition *is* PyTorch fluency.